# Prototypage

Nos essais avant d'écrire le package. Ici le code est volontairement brouillon : la version propre de chaque idée est dans `log_engine/`. Ce notebook n'est pas utilisé par `main.py`.

In [1]:
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import NamedTuple

if Path.cwd().name == "notebooks":
    os.chdir("..")

with open("data/app_logs_1.log", encoding="utf-8", errors="replace") as f:
    sample = [next(f).rstrip("\n") for _ in range(8)]
for line in sample:
    print(line)

2026-11-04T05:50:03 | ERROR | 192.168.1.18:22/SSH | user=u008 | {"event": "login_failed", "level": "ERROR", "code": 401}
2026-11-04T05:50:06 | WARNING | 192.168.1.35:22/TCP | user=u003 | Temps de réponse élevé
2026-11-04T05:50:09 | ERROR | 192.168.1.35:22/SSH | user=u014 | {"event": "login_failed", "level": "ERROR", "code": 401}
2026-11-04T05:50:12 | INFO | 192.168.1.1:443/UDP | user=u025 | Fichier téléchargé
2026-11-04T05:50:14 | INFO | 192.168.1.23:22/SSH | user=u020 | {"event": "login_success", "level": "INFO"}
2026-11-04T05:50:15 | INFO | 192.168.1.35:80/UDP | user=u004 | Session ouverte
2026-11-04T05:50:17 | INFO | 192.168.1.5:80/TCP | user=u002 | Fichier téléchargé
2026-11-04T05:50:20 | INFO | 192.168.1.11:22/SSH | user=u012 | {"event": "login_success", "level": "INFO"}


## Essai 1 : découper une ligne

Première idée : `split(" | ")`. Ça marche sur une ligne propre, mais on n'a aucun contrôle : une ligne cassée donne un nombre de morceaux au hasard, et l'IP, le port et le protocole restent collés.

In [2]:
parts = sample[0].split(" | ")
print(len(parts), "morceaux :", parts)
print("Ligne cassée :", "### ligne corrompue ###".split(" | "))

5 morceaux : ['2026-11-04T05:50:03', 'ERROR', '192.168.1.18:22/SSH', 'user=u008', '{"event": "login_failed", "level": "ERROR", "code": 401}']
Ligne cassée : ['### ligne corrompue ###']


Deuxième idée : une regex avec des groupes nommés. Une ligne au mauvais format ne correspond pas (`None`), et on récupère directement chaque champ par son nom.

In [3]:
import re

pattern = re.compile(
    r"^(?P<timestamp>\S+) \| (?P<level>\w+) \| "
    r"(?P<ip>[\d.]+):(?P<port>\d+)/(?P<protocol>\w+) \| "
    r"user=(?P<user>\w+) \| (?P<payload>.+)$"
)
m = pattern.match(sample[0])
print(m.groupdict())
print("Ligne cassée :", pattern.match("### ligne corrompue ###"))

{'timestamp': '2026-11-04T05:50:03', 'level': 'ERROR', 'ip': '192.168.1.18', 'port': '22', 'protocol': 'SSH', 'user': 'u008', 'payload': '{"event": "login_failed", "level": "ERROR", "code": 401}'}
Ligne cassée : None


**Retenu :** la regex, dans `config.py` (`LINE_PATTERN`).

## Essai 2 : liste ou générateur ?

In [4]:
def lire_liste(path):
    with open(path, encoding="utf-8") as f:
        return f.readlines()


def lire_generateur(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line


liste = lire_liste("data/app_logs_1.log")
gen = lire_generateur("data/app_logs_1.log")
print("liste      :", sys.getsizeof(liste), "octets (+ le texte de chaque ligne)")
print("générateur :", sys.getsizeof(gen), "octets")
print("première ligne du générateur :", next(gen)[:40])

liste      : 8856 octets (+ le texte de chaque ligne)
générateur : 232 octets
première ligne du générateur : 2026-11-04T05:50:03 | ERROR | 192.168.1.


**Retenu :** le générateur (`read_log_file`). Sa taille ne dépend pas du fichier.

Pour plusieurs fichiers, on a d'abord écrit une double boucle, puis on a vu que `yield from` fait la même chose en une ligne :

In [5]:
def plusieurs_v1(paths):
    for path in paths:
        for line in lire_generateur(path):
            yield line


def plusieurs_v2(paths):
    for path in paths:
        yield from lire_generateur(path)


files = ["data/app_logs_1.log", "data/app_logs_2.log"]
print(sum(1 for _ in plusieurs_v1(files)), sum(1 for _ in plusieurs_v2(files)))

2024 2024


## Essai 3 : `match-case`, et le bug de l'ordre des cas

Premier jet : on a mis `event` avant `error`.

In [6]:
def route_v1(item):
    match item:
        case str():
            return "raw"
        case {"event": str()}:
            return "event"
        case {"level": "ERROR" | "CRITICAL", "code": int()}:
            return "error"
        case _:
            return "unknown"


echec = {"event": "login_failed", "level": "ERROR", "code": 401}
print("v1 :", route_v1(echec))

v1 : event


Bug : un échec de connexion contient **aussi** la clé `event`, donc il tombe dans le premier cas qui correspond, `event`, et les erreurs ne sont jamais comptées. Python teste les `case` de haut en bas et s'arrête au premier qui marche. On inverse :

In [7]:
def route_v2(item):
    match item:
        case str():
            return "raw"
        case {"level": "ERROR" | "CRITICAL", "code": int()}:
            return "error"
        case {"event": str()}:
            return "event"
        case (str(), int(), "TCP" | "UDP" | "SSH"):
            return "connection"
        case _:
            return "unknown"


print("v2 :", route_v2(echec))
print("tuple :", route_v2(("1.2.3.4", 22, "SSH")), "|", route_v2(("1.2.3.4", 21, "FTP")))

v2 : error
tuple : connection | unknown


**Retenu :** `route_v2`, devenu `route` dans `pipeline.py`. On a gardé un test pour ce cas précis.

## Essai 4 : tuple simple ou `NamedTuple` pour la connexion ?

In [8]:
simple = ("192.168.1.18", 22, "SSH")
print("tuple simple, port :", simple[1], "(il faut se souvenir que c'est la case 1)")


class ConnectionMeta(NamedTuple):
    ip: str
    port: int
    protocol: str


nomme = ConnectionMeta("192.168.1.18", 22, "SSH")
print("NamedTuple, port :", nomme.port, "| reste un tuple :", isinstance(nomme, tuple))
print("route_v2 le reconnaît toujours :", route_v2(nomme))
try:
    nomme.port = 80
except AttributeError as error:
    print("immuable :", error)

tuple simple, port : 22 (il faut se souvenir que c'est la case 1)
NamedTuple, port : 22 | reste un tuple : True
route_v2 le reconnaît toujours : connection
immuable : can't set attribute


**Retenu :** le `NamedTuple` : aussi immuable qu'un tuple, mais lisible.

## Essai 5 : compter avec un dict ou un `defaultdict` ?

In [9]:
levels = [line.split(" | ")[1] for line in sample]

compteur = {}
try:
    for level in levels:
        compteur[level] += 1
except KeyError as error:
    print("dict normal : KeyError", error)

compteur = defaultdict(int)
for level in levels:
    compteur[level] += 1
print("defaultdict :", dict(compteur))

dict normal : KeyError 'ERROR'
defaultdict : {'ERROR': 2, 'WARNING': 1, 'INFO': 5}


**Retenu :** `defaultdict(int)` partout dans `aggregate`.

## Essai 6 : la force brute, et le `else` de boucle

On voulait savoir si le `else` d'une boucle sert vraiment ici. Il s'exécute seulement quand la boucle n'a pas été coupée par `break` : c'est exactement « l'IP a été contrôlée jusqu'au bout sans attaque ».

In [10]:
def verifie(tentatives, seuil):
    serie = 0
    for echec in tentatives:
        serie = serie + 1 if echec else 0
        if serie >= seuil:
            resultat = "attaque"
            break
    else:
        resultat = "propre"
    return resultat


print("8 échecs        :", verifie([True] * 8, 5))
print("4 échecs + 1 ok :", verifie([True] * 4 + [False], 5), "(l'IP piège)")
print("aucun échec     :", verifie([False] * 6, 5))

8 échecs        : attaque
4 échecs + 1 ok : propre (l'IP piège)
aucun échec     : propre


**Retenu :** la même logique dans `detect_brute_force`, avec en plus `continue` pour sauter les IP qui ont moins de tentatives que le seuil, et une boucle `while` sur les tentatives comme demandé par le sujet. C'est cette IP piège qui nous a donné l'idée de cacher des attaques à des positions connues dans le générateur de logs.